# Plan with LeWM on OGBCubeDR (RunPod)

Runs `scripts/plan/eval_wm.py` against a trained LeWM checkpoint on `cube_quadruple_dr_expert`: first the **original single-goal** planning eval (`num_subgoals=1`, the planner as it existed before this project), then the **full-task chained-subgoal** eval this project added (`num_subgoals=8` by default, see `Run.md` §7 / §1). Flow: config → apply → GPU check → install → verify assets → pick checkpoint epoch → run both evals → compare results/videos.

Prerequisite: a trained checkpoint under `$STABLEWM_HOME/checkpoints/<OUTPUT_MODEL_NAME>/` (e.g. produced by `train_lewm_ogbcubedr.ipynb` on this same volume) and the `cube_quadruple_dr_expert` dataset under `$STABLEWM_HOME/datasets/ogbench/`. If this pod has the same `/workspace` network volume used for training, both are already there and nothing needs downloading.

## 1. Config

Edit the values below.

In [ ]:
import os

# --- repo ---
REPO_ROOT = '/workspace/stable-worldmodel'          # ← edit if you cloned it elsewhere

# --- storage (network volume) ---
STABLEWM_HOME = '/workspace'                        # datasets/, checkpoints/ live directly here

# --- checkpoint to evaluate ---
OUTPUT_MODEL_NAME = 'lewm_q4_dr'                    # matches the training run name (baseline_lewm_08_2026/checkpoints/lewm_q4_dr locally)
POLICY_EPOCH = None                                 # ← set an int to pin an epoch; None = auto-pick the latest weights_epoch_N.pt on this volume

# --- fallback: only used if the checkpoint/dataset are NOT already on this volume ---
HF_TOKEN = os.environ.get('HF_TOKEN', '')                                    # ← paste here if not set as a pod env var
HF_DATASET_REPO_ID = '<your-hf-username-or-org>/ogbench-cube-quadruple-domain-randomized-expert'  # ← edit me
HF_CHECKPOINT_REPO_ID = ''                          # ← edit me if the checkpoint also lives on HF; leave '' to skip

# --- renderer (headless GPU pod; see Run.md §3 table for CPU-only pods) ---
MUJOCO_GL = 'egl'                                   # eval_wm.py's own default; osmesa + PYOPENGL_PLATFORM=osmesa on CPU-only pods

# --- eval scale (cfg.eval.num_eval — episodes AND parallel MuJoCo envs; lower for a fast first pass) ---
NUM_EVAL_EPISODES = 50                              # shipped config default

# --- derived, don't edit ---
DATASET_DIR = os.path.join(STABLEWM_HOME, 'datasets', 'ogbench', 'cube_quadruple_dr_expert.lance')
CHECKPOINT_DIR = os.path.join(STABLEWM_HOME, 'checkpoints', OUTPUT_MODEL_NAME)
RESULTS_ROOT = os.path.join(STABLEWM_HOME, 'planning_results', OUTPUT_MODEL_NAME)  # this notebook's own tidy copy of outputs


## 2. Apply config

In [ ]:
os.environ['STABLEWM_HOME'] = STABLEWM_HOME
os.environ['MUJOCO_GL'] = MUJOCO_GL
os.environ['HF_TOKEN'] = HF_TOKEN

os.makedirs(STABLEWM_HOME, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)

os.chdir(REPO_ROOT)  # os.chdir (not `%cd`) so it's identical whether run fresh or after a kernel restart

print('cwd            =', os.getcwd())
print('STABLEWM_HOME  =', os.environ['STABLEWM_HOME'])
print('MUJOCO_GL      =', os.environ['MUJOCO_GL'])
print('DATASET_DIR    =', DATASET_DIR)
print('CHECKPOINT_DIR =', CHECKPOINT_DIR)
!df -h /workspace


## 3. GPU check

`eval_wm.py` defaults `MUJOCO_GL=egl`, which needs a GPU (see config cell if this is a CPU-only pod).

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU visible — MUJOCO_GL=egl needs one; set MUJOCO_GL=osmesa in the config cell for a CPU pod'
print(torch.cuda.get_device_name(0))


## 4. Install dependencies

Unlike training, eval **renders** — so the system GL libraries from `Run.md` §2 are required here, not just the pip packages. A bare `nvidia/cuda` or `pytorch` image ships none of them, and without `libEGL` PyOpenGL's EGL platform loads as `None` and `import mujoco` dies with `AttributeError: 'NoneType' object has no attribute 'eglQueryString'`.

The pip install is the same scoped one as training — see `Run.md` §3 for why not `.[all]`.

In [ ]:
# System GL libraries (Run.md §2 container list). Root in a pod, so no sudo;
# DEBIAN_FRONTEND=noninteractive keeps apt from hanging on the tzdata prompt.
# Covers both renderers: libegl1/libopengl0 for MUJOCO_GL=egl, libosmesa6-dev for osmesa.
!apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
    libgl1-mesa-dev libgl1 libglx-mesa0 libglfw3 \
    libosmesa6-dev libegl1 libopengl0 patchelf

!ldconfig -p | grep -E 'libEGL\.so|libOSMesa\.so' || echo 'WARNING: still no libEGL/libOSMesa on the loader path'


In [ ]:
%pip install -q -e '.[train,format]'
%pip install -q ogbench pygame pymunk shapely opencv-python-headless gymnasium-robotics huggingface_hub


In [ ]:
# Renderer smoke test — same subprocess + env that eval_wm.py will get, so a broken
# GL context fails here in a second instead of after the model and dataset have loaded.
# Run in a subprocess: a failed GL context can leave this kernel unable to retry.
import subprocess
import sys
import textwrap

_probe = textwrap.dedent("""
    import os
    import mujoco
    m = mujoco.MjModel.from_xml_string('<mujoco><worldbody/></mujoco>')
    r = mujoco.Renderer(m, 64, 64)
    r.update_scene(mujoco.MjData(m))
    r.render()
    print('renderer OK — MUJOCO_GL=' + os.environ.get('MUJOCO_GL', '<unset>'))
""")

_p = subprocess.run([sys.executable, '-c', _probe], capture_output=True, text=True)
print(_p.stdout.strip() or _p.stderr.strip()[-1500:])
assert _p.returncode == 0, (
    f'MUJOCO_GL={MUJOCO_GL} cannot create a context on this pod.\n'
    "If the apt cell succeeded, this pod likely wasn't started with graphics driver "
    "capabilities (NVIDIA_DRIVER_CAPABILITIES must include 'graphics' for EGL, and it "
    'is set at pod-creation time only). Set MUJOCO_GL = "osmesa" in the config cell and '
    're-run from §2 — software rendering, slower but always available.'
)


## 5. Verify (or fetch) the dataset and checkpoint

Both should already be on this volume if it's the same one `train_lewm_ogbcubedr.ipynb` used. Falls back to an HF download only if missing — edit `HF_DATASET_REPO_ID` / `HF_CHECKPOINT_REPO_ID` in the config cell first.

In [ ]:
if os.path.isdir(DATASET_DIR) and os.listdir(DATASET_DIR):
    print('Dataset already present:', DATASET_DIR)
else:
    assert HF_DATASET_REPO_ID and '<' not in HF_DATASET_REPO_ID, (
        'Dataset missing and HF_DATASET_REPO_ID is not set — edit the config cell.'
    )
    os.makedirs(DATASET_DIR, exist_ok=True)
    !hf download "$HF_DATASET_REPO_ID" --repo-type dataset --local-dir "$DATASET_DIR"


In [ ]:
if os.path.isdir(CHECKPOINT_DIR) and any(f.endswith('.pt') for f in os.listdir(CHECKPOINT_DIR)):
    print('Checkpoint already present:', CHECKPOINT_DIR)
else:
    assert HF_CHECKPOINT_REPO_ID, (
        f'{CHECKPOINT_DIR} has no weights_epoch_*.pt and HF_CHECKPOINT_REPO_ID is not set.\n'
        'Either copy the checkpoint onto this volume (e.g. from baseline_lewm_08_2026/checkpoints/) '
        'or set HF_CHECKPOINT_REPO_ID in the config cell.'
    )
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    !hf download "$HF_CHECKPOINT_REPO_ID" --local-dir "$CHECKPOINT_DIR"

assert os.path.exists(os.path.join(CHECKPOINT_DIR, 'config.json')), (
    f'{CHECKPOINT_DIR} is missing config.json — load_pretrained() needs it next to the .pt file.'
)


## 6. Pick the checkpoint epoch

`load_pretrained` needs one specific `.pt` file — it errors on an ambiguous folder with several. Auto-picks the latest epoch on this volume unless `POLICY_EPOCH` is set above.

In [ ]:
import glob
import re

ckpt_files = glob.glob(os.path.join(CHECKPOINT_DIR, 'weights_epoch_*.pt'))
assert ckpt_files, f'No weights_epoch_*.pt found in {CHECKPOINT_DIR}'

epochs = sorted(int(re.search(r'weights_epoch_(\d+)\.pt$', f).group(1)) for f in ckpt_files)
epoch = POLICY_EPOCH if POLICY_EPOCH is not None else epochs[-1]
assert epoch in epochs, f'weights_epoch_{epoch}.pt not found; available epochs: {epochs}'

POLICY = f'{OUTPUT_MODEL_NAME}/weights_epoch_{epoch}.pt'
print('Available epochs:', epochs)
print('Using policy     :', POLICY)


## 7. Two planning evals

`scripts/plan/eval_wm.py` (config: `scripts/plan/config/cube_quadruple_dr.yaml`) replays a recorded expert episode, restores the sim to that step (including its sampled domain randomization), then hands the planner a chain of oracle subgoals (`Run.md` §7):

- **Original planning** — `eval.num_subgoals=1`: a single fixed goal, the planner's behavior before this project. `Run.md` calls this the check that must "reproduce the old single-goal numbers" before trusting the chain.
- **Full-task planning** — the project-specific addition (`Run.md` §1, `stable_worldmodel/world/world.py`): a chain of `num_subgoals` oracle subgoals spaced `goal_offset_steps` apart; success is judged against the **final** subgoal, i.e. whether the whole replayed segment was completed. Left at the shipped defaults (`num_subgoals=8`, `eval_budget=320`).

**Gotcha, handled below**: for a real (non-random) policy, `eval_wm.py` always writes videos to `$STABLEWM_HOME/checkpoints/<policy>/env_{i}.mp4` — a fixed path that doesn't depend on `hydra.run.dir`. Running both evals back to back would overwrite the first run's videos with the second's, so each run below is immediately followed by a cell that moves its outputs into a labeled folder under `RESULTS_ROOT` before the next run starts. `output.filename=` *is* overridable, so each run also gets its own results file.

In [ ]:
!python scripts/plan/eval_wm.py --config-name=cube_quadruple_dr \
    policy="$POLICY" \
    eval.num_eval=$NUM_EVAL_EPISODES \
    eval.num_subgoals=1 eval.eval_budget=50 \
    output.filename=original_results.txt


In [ ]:
import shutil

original_dir = os.path.join(RESULTS_ROOT, f'original_epoch{epoch}')
os.makedirs(original_dir, exist_ok=True)

for f in glob.glob(os.path.join(CHECKPOINT_DIR, 'env_*.mp4')) + [os.path.join(CHECKPOINT_DIR, 'original_results.txt')]:
    if os.path.exists(f):
        shutil.move(f, os.path.join(original_dir, os.path.basename(f)))

print('Original-planning outputs moved to', original_dir)
print(sorted(os.listdir(original_dir)))


Full-task (chained oracle subgoals) — shipped defaults, no overrides needed:

In [ ]:
!python scripts/plan/eval_wm.py --config-name=cube_quadruple_dr \
    policy="$POLICY" \
    eval.num_eval=$NUM_EVAL_EPISODES \
    output.filename=full_task_results.txt


In [ ]:
full_task_dir = os.path.join(RESULTS_ROOT, f'full_task_epoch{epoch}')
os.makedirs(full_task_dir, exist_ok=True)

for f in glob.glob(os.path.join(CHECKPOINT_DIR, 'env_*.mp4')) + [os.path.join(CHECKPOINT_DIR, 'full_task_results.txt')]:
    if os.path.exists(f):
        shutil.move(f, os.path.join(full_task_dir, os.path.basename(f)))

print('Full-task outputs moved to', full_task_dir)
print(sorted(os.listdir(full_task_dir)))


## 8. (Optional) random baseline for reference

`policy=random` writes next to the script (`scripts/plan/`, not the checkpoint dir — `Run.md`'s own troubleshooting note), so it's collected from a different source path.

In [ ]:
RUN_RANDOM_BASELINE = True  # ← set False to skip

random_dir = os.path.join(RESULTS_ROOT, 'random_baseline')

if RUN_RANDOM_BASELINE:
    !python scripts/plan/eval_wm.py --config-name=cube_quadruple_dr \
        policy=random \
        eval.num_eval=$NUM_EVAL_EPISODES \
        output.filename=random_results.txt

    script_dir = os.path.join(REPO_ROOT, 'scripts', 'plan')
    os.makedirs(random_dir, exist_ok=True)
    for f in glob.glob(os.path.join(script_dir, 'env_*.mp4')) + [os.path.join(script_dir, 'random_results.txt')]:
        if os.path.exists(f):
            shutil.move(f, os.path.join(random_dir, os.path.basename(f)))
    print('Random-baseline outputs moved to', random_dir)


## 9. Compare results

In [ ]:
runs = {'original (num_subgoals=1)': original_dir, 'full task (chained subgoals)': full_task_dir}
if RUN_RANDOM_BASELINE:
    runs['random baseline'] = random_dir

for label, run_dir in runs.items():
    txt_files = glob.glob(os.path.join(run_dir, '*_results.txt'))
    print(f'=== {label} ===')
    if not txt_files:
        print(f'  no results file in {run_dir}')
        print()
        continue
    text = open(txt_files[0]).read()
    results_section = text.split('==== RESULTS ====')[-1]
    print(results_section.strip())
    print()


## 10. Watch a video

Each run writes one `env_{i}.mp4` per episode — three labelled panels: `agent | dataset | goal`. The goal panel advances as the subgoal chain does.

In [ ]:
from IPython.display import Video, display

for label, run_dir in runs.items():
    vids = sorted(glob.glob(os.path.join(run_dir, 'env_*.mp4')))
    if not vids:
        print(f'{label}: no videos found in {run_dir}')
        continue
    print(label, '-', vids[0])
    display(Video(vids[0], embed=True, width=640))


## Next: sweep subgoal density

Same fixed total budget, increasing chain length (`Run.md` §7), via Hydra multirun:

```bash
python scripts/plan/eval_wm.py -m --config-name=cube_quadruple_dr \
    policy=lewm_q4_dr/weights_epoch_<N>.pt \
    eval.num_subgoals=1,2,4,8 eval.eval_budget=320
```

Multirun runs each combination sequentially in-process and shares the same output-path collision behavior as above — run it from a pod terminal rather than this notebook, and move `env_*.mp4` out between combinations if you want to keep every sweep point's videos.